In [45]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

# Load Dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Convert labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Training shape:", x_train.shape)
print("Test shape:", x_test.shape)

Training shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


#### Base MLP Model

In [46]:
def create_base_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(32,32,3)),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

#### L2 Regularization

In [47]:
def create_l2_model():
    model = models.Sequential([
        layers.Flatten(input_shape=(32,32,3)),
        layers.Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
        layers.Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

#### Dataset Augmentation

In [48]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

def create_aug_model():
    model = models.Sequential([
        data_augmentation,
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

#### Parameter Sharing & Tying

In [49]:
def create_shared_model():
    
    inputs = tf.keras.Input(shape=(32,32,3))
    x = layers.Flatten()(inputs)           # (None, 3072)
    
    # First projection layer
    x = layers.Dense(256, activation='relu')(x)  
    
    # Shared layer (256 -> 256)
    shared_layer = layers.Dense(256, activation='relu')
    
    x = shared_layer(x)
    x = shared_layer(x)   # Reuse safely (same input size)
    
    outputs = layers.Dense(10, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

#### Adding Noise to Inputs

In [50]:
def create_noise_model():
    model = models.Sequential([
        layers.GaussianNoise(0.1),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

#### Dropout

In [51]:
def create_dropout_model():
    model = models.Sequential([
        layers.Flatten(input_shape=(32,32,3)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

#### Early Stopping

In [52]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model = create_base_model()
history_es = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)

print("Final Training Accuracy:", history_es.history['accuracy'][-1])
print("Final Validation Accuracy:", history_es.history['val_accuracy'][-1])

Final Training Accuracy: 0.5793250203132629
Final Validation Accuracy: 0.4948999881744385


#### Ensemble Method

In [53]:
def train_ensemble(n_models=3):
    models_list = []
    
    for i in range(n_models):
        model = create_base_model()
        model.fit(x_train, y_train,
                  epochs=10,
                  batch_size=64,
                  verbose=0)
        models_list.append(model)
    
    return models_list

def evaluate_ensemble(models_list):
    predictions = [model.predict(x_test) for model in models_list]
    avg_pred = np.mean(predictions, axis=0)
    final_pred = np.argmax(avg_pred, axis=1)
    true_labels = np.argmax(y_test, axis=1)
    
    accuracy = np.mean(final_pred == true_labels)
    return accuracy

In [54]:
models_dict = {
    "Base": create_base_model(),
    "L2": create_l2_model(),
    "Augmentation": create_aug_model(),
    "Dropout": create_dropout_model(),
    "Noise": create_noise_model(),
    "Parameter Sharing": create_shared_model()
}

results = {}

for name, model in models_dict.items():
    print(f"\nTraining {name}")
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=40,
        batch_size=64,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    results[name] = test_acc
    print(f"{name} Test Accuracy: {test_acc:.4f}")

print("\n===== Final Comparison =====")
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

# Ensemble
ensemble_models = train_ensemble()
ensemble_acc = evaluate_ensemble(ensemble_models)
print(f"\nEnsemble Accuracy: {ensemble_acc:.4f}")


Training Base
Base Test Accuracy: 0.4815

Training L2
L2 Test Accuracy: 0.4963

Training Augmentation
Augmentation Test Accuracy: 0.4961

Training Dropout
Dropout Test Accuracy: 0.4180

Training Noise
Noise Test Accuracy: 0.4996

Training Parameter Sharing
Parameter Sharing Test Accuracy: 0.4614

===== Final Comparison =====
Base: 0.4815
L2: 0.4963
Augmentation: 0.4961
Dropout: 0.4180
Noise: 0.4996
Parameter Sharing: 0.4614
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Ensemble Accuracy: 0.5340
